# 02. Policy Gradient Reinforce A2c PPO and GFlowNets

- **Category:** 08_learning_paradigms_rl_and_alignment / 02_reinforcement_learning_and_world_models
- **Lineage & Citations:** Williams (1992 REINFORCE); Sutton et al. (1999 Policy Gradient Theorem); Schulman et al. (2015 GAE, 2017 PPO); Bengio et al. (2021 GFlowNets); Malkin et al. (2022 Trajectory Balance).
- **Purpose:** First-principles mathematical implementation and empirical verification of policy gradient reinforcement learning methods, from REINFORCE and Generalized Advantage Estimation (GAE) to PPO-Clip and Generative Flow Networks (GFlowNets).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. Value Based RL Dqn Rainbow and Qr Dqn](01_value_based_rl_dqn_rainbow_and_qr_dqn.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. Continuous Control Ddpg Td3 and SAC](03_continuous_control_ddpg_td3_and_sac.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never compute the PPO probability ratio $r_t = \pi_\theta / \pi_{\text{old}}$ directly in probability space; on high-dimensional action spaces, small probabilities cause catastrophic floating-point underflow ($0.0 / 0.0 \to \text{NaN}$). Always use log space: $\exp(\log \pi_\theta - \log \pi_{\text{old}})$. Do NOT use reward-maximizing policy gradients (like PPO) when the goal is discovering diverse functional candidates (e.g., drug candidates, crystal structures, formal proofs); PPO inevitably mode-collapses onto the single highest-reward peak, whereas GFlowNets sample candidates proportionally to reward: $P(x) \propto R(x)$.

## Mathematical Foundations & Governing Equations

### 1. Generalized Advantage Estimation (GAE)
$$
\delta_t^V = r_t + \gamma (1 - d_t) V(s_{t+1}) - V(s_t)
$$
$$
\hat{A}_t^{\text{GAE}(\gamma, \lambda)} = \sum_{l=0}^\infty (\gamma \lambda)^l (1 - d_{t+l}) \delta_{t+l}^V = \delta_t^V + \gamma \lambda (1 - d_t) \hat{A}_{t+1}^{\text{GAE}}
$$

### 2. Proximal Policy Optimization (PPO-Clip)
$$
r_t(\theta) = \exp\left( \log \pi_\theta(a_t \mid s_t) - \log \pi_{\theta_{\text{old}}}(a_t \mid s_t) \right)
$$
$$
\mathcal{L}_{\text{CLIP}}(\theta) = -\hat{\mathbb{E}}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) \hat{A}_t \right) \right]
$$

### 3. GFlowNet Trajectory Balance (TB)
$$
\mathcal{L}_{\text{TB}}(\tau) = \left( \log Z_\theta + \sum_{t=0}^{T-1} \log P_F(s_{t+1} \mid s_t; \theta) - \log R(x) - \sum_{t=0}^{T-1} \log P_B(s_t \mid s_{t+1}; \theta) \right)^2
$$

### Asymptotic Time & Memory Complexity
- GAE Trajectory Scan: $\mathcal{O}(T)$ sequential time backward, $\mathcal{O}(T)$ memory.
- PPO-Clip Surrogate Loss: $\mathcal{O}(B)$ elementwise time and memory.
- GFlowNet Trajectory Balance: $\mathcal{O}(B \cdot T)$ time and memory for batch size $B$ and trajectory length $T$.

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/02_policy_gradient_reinforce_a2c_ppo_and_gflownets.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def ppo_clip_loss(
    log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
    advantages: torch.Tensor,
    clip_eps: float = 0.2,
) -> torch.Tensor:
    ratio = torch.exp(log_probs - old_log_probs)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages
    clipped_surrogate = torch.min(surr1, surr2)
    return -torch.mean(clipped_surrogate)

def compute_gae(
    rewards: torch.Tensor,
    values: torch.Tensor,
    dones: torch.Tensor,
    next_value: torch.Tensor,
    gamma: float = 0.99,
    lambda_: float = 0.95,
) -> torch.Tensor:
    t_len = rewards.shape[0]
    advantages = torch.zeros(t_len, dtype=torch.float32, device=rewards.device)
    last_gae = torch.tensor(0.0, device=rewards.device)

    for t in reversed(range(t_len)):
        v_next = next_value if t == t_len - 1 else values[t + 1]
        not_done = 1.0 - dones[t]
        delta = rewards[t] + gamma * not_done * v_next - values[t]
        last_gae = delta + gamma * lambda_ * not_done * last_gae
        advantages[t] = last_gae

    return advantages

def gflownet_trajectory_balance_loss(
    log_z: torch.Tensor,
    log_pf_trajectory: torch.Tensor,
    log_pb_trajectory: torch.Tensor,
    reward: torch.Tensor,
) -> torch.Tensor:
    sum_log_pf = torch.sum(log_pf_trajectory, dim=-1)
    sum_log_pb = torch.sum(log_pb_trajectory, dim=-1)
    log_r = torch.log(torch.clamp(reward, min=1e-8))
    tb_error = (log_z + sum_log_pf) - (log_r + sum_log_pb)
    return torch.mean(tb_error ** 2)

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-6, f"Linear anchor parity error: {err_anchor:.2e}"

ppo_l = ppo_clip_loss(weights["log_probs"], weights["old_log_probs"], weights["advantages"], clip_eps=0.2)
err_ppo = float(torch.max(torch.abs(ppo_l - weights["ppo_loss_ref"])))
assert err_ppo < 1e-6, f"PPO loss parity error: {err_ppo:.2e}"

gae_adv = compute_gae(weights["rewards"], weights["values"], weights["dones"], weights["next_value"], gamma=0.99, lambda_=0.95)
err_gae = float(torch.max(torch.abs(gae_adv - weights["gae_advantages_ref"])))
assert err_gae < 1e-6, f"GAE advantages parity error: {err_gae:.2e}"

tb_l = gflownet_trajectory_balance_loss(weights["log_z"], weights["log_pf"], weights["log_pb"], weights["reward"])
err_tb = float(torch.max(torch.abs(tb_l - weights["tb_loss_ref"])))
assert err_tb < 1e-6, f"TB loss parity error: {err_tb:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_ppo, err_gae, err_tb):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_ppo = benchmark_fn(
    lambda: ppo_clip_loss(weights["log_probs"], weights["old_log_probs"], weights["advantages"]),
    iters=100,
    name="PPO-Clip Loss",
)
b_gae = benchmark_fn(
    lambda: compute_gae(weights["rewards"], weights["values"], weights["dones"], weights["next_value"]),
    iters=100,
    name="GAE Trajectory Scan",
)
b_tb = benchmark_fn(
    lambda: gflownet_trajectory_balance_loss(weights["log_z"], weights["log_pf"], weights["log_pb"], weights["reward"]),
    iters=100,
    name="GFlowNet Trajectory Balance",
)

print(format_benchmark_table([b_ppo, b_gae, b_tb]))

In [ ]:
# Failure Mode Contrast: Unclipped Policy Gradient vs PPO-Clip Destructive Step
# Suppose a rogue gradient update yields a massive policy shift (r_t = 10.0) with positive advantage A_t = 2.0
ratio_extreme = torch.tensor([10.0])
advantage_pos = torch.tensor([2.0])

# Unclipped policy surrogate objective:
unclipped_obj = (ratio_extreme * advantage_pos).item()

# PPO clipped policy surrogate objective (eps = 0.2, max allowed ratio 1.2):
clipped_ratio = torch.clamp(ratio_extreme, 1.0 - 0.2, 1.0 + 0.2)
ppo_clipped_obj = torch.min(ratio_extreme * advantage_pos, clipped_ratio * advantage_pos).item()

print(f"Unclipped Objective Magnitude: {unclipped_obj:.2f} (Explosive policy update!)")
print(f"PPO Clipped Objective Magnitude: {ppo_clipped_obj:.2f} (Strictly bounded to (1+eps)*A = 2.40)")

assert abs(unclipped_obj - 20.0) < 1e-4, "Unclipped update must reflect full ratio scaling"
assert abs(ppo_clipped_obj - 2.4) < 1e-4, "PPO clip must bound objective strictly at 2.4"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Pessimistic Clipping**: Taking $\min(r_t A_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t)$ acts as a lower bound on improvement, preventing destructive policy divergence.
2. **GAE Bias-Variance Duality**: The parameter $\lambda$ continuously interpolates between 1-step TD ($\,\lambda=0$, low variance, biased) and full Monte Carlo ($\,\lambda=1$, unbiased, high variance).
3. **Reward-Proportional Sampling vs Maximization**: GFlowNets replace greedy mode-seeking with flow-conserving probability matching, turning generation into a principled exploration engine.

---
### Cross-Domain Generalisation Matrix
- **Molecular Synthesis**: GFlowNet Trajectory Balance samples diverse molecular graphs covering multiple distant binding affinity peaks rather than repeatedly synthesizing variants of one chemical structure.
- **Robotics Locomotion**: PPO with GAE provides steady policy improvement under unmodeled terrain variations without requiring exact simulator dynamic matrices.
- **Algorithmic Execution**: Clipped policy objectives prevent sudden high-volume order book dumping caused by transient liquidity spikes.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_policy_gradient_reinforce_a2c_ppo_and_gflownets.md](../../../markdown/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/02_policy_gradient_reinforce_a2c_ppo_and_gflownets.md)
- **PyTorch 2.x Module:** [02_policy_gradient_reinforce_a2c_ppo_and_gflownets.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/02_policy_gradient_reinforce_a2c_ppo_and_gflownets.py)
- **Functional JAX Module:** [02_policy_gradient_reinforce_a2c_ppo_and_gflownets.py](../../../jax/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/02_policy_gradient_reinforce_a2c_ppo_and_gflownets.py)
- **Native Rust Candle Track:** [02_policy_gradient_reinforce_a2c_ppo_and_gflownets.rs](../../../candle/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/02_policy_gradient_reinforce_a2c_ppo_and_gflownets.rs)
- **Hardware Kernel (CPU / WGSL):** [02_policy_gradient_reinforce_a2c_ppo_and_gflownets.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/02_policy_gradient_reinforce_a2c_ppo_and_gflownets.rs)
- **Golden SafeTensors:** [02_policy_gradient_reinforce_a2c_ppo_and_gflownets.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/02_policy_gradient_reinforce_a2c_ppo_and_gflownets.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Value Based RL Dqn Rainbow and Qr Dqn](01_value_based_rl_dqn_rainbow_and_qr_dqn.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 03. Continuous Control Ddpg Td3 and SAC](03_continuous_control_ddpg_td3_and_sac.ipynb) &rarr;